# Jais Arabic Voice Chatbot — استضافة على كولاب 🇸🇦🎙️

كل شيء يعمل **على جهاز كولاب الافتراضي (T4 في السحابة)** — لا شيء على جهازك.

`المتصفح ← WebRTC ← Jais-ASR (STT) ← Jais-3 (LLM) ← Audar-TTS (TTS)`

**مهم:** خلية 4 لازم تخلص **بلا أي ❌** — باستخدام توكن Read بعد قبول ترخيص [neuphonic/neucodec](https://huggingface.co/neuphonic/neucodec).

**الخطوات:**
1. `Runtime → Change runtime type → GPU (T4)`
2. `Run all` (أول مرة ≈ 10 دقائق: pip + ~7 GB نماذج)
3. الخلية 6: 🌍 رابط استضافة عام (يتحقق منه آليًا) — شاركه/افتحه من أي جهاز
4. أي خلل → خلية 7 — والسجلات تطبع تلقائيًا عند أي فشل

[&#8593; افتح هذا الدفتر في Colab ](https://colab.research.google.com/github/911houssem/jais-voice-chatbot/blob/main/notebooks/jais_voice_chatbot_colab.ipynb)

In [ ]:
#@title 1) فحص كرت الشاشة واختيار الجودة المناسبة {display-mode:"form"}
import subprocess, re

def gpu_state():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
             "--format=csv,noheader"]).decode()
        name, total, free = [x.strip() for x in out.split(",")]
        return name, int(re.sub(r"\D", "", total)) / 1024.0, int(re.sub(r"\D", "", free)) / 1024.0
    except Exception as e:
        print("No GPU detected:", e)
        return "", 0.0, 0.0

name, total_gb, free_gb = gpu_state()
print(f"GPU: {name} | total {total_gb:.1f} GB | free {free_gb:.1f} GB")
if free_gb and free_gb < 20:
    print("Less than 20 GB free -> JAIS_ASR_MODEL=auto picks the W4A16-AWQ ASR checkpoint.")
else:
    print("20+ GB free -> JAIS_ASR_MODEL=auto picks the fp16 Jais-ASR checkpoint.")

In [ ]:
#@title 2) استنساخ المشروع وتثبيت الاعتمادات + فحص pipecat تلقائي (~5 دقائق) {display-mode:"form"}
REPO_URL = "https://github.com/911houssem/jais-voice-chatbot.git" #@param {type:"string"}

import os, subprocess, sys, glob, shutil, sysconfig
repo_dir = "/content/jais-voice-chatbot"
if not os.path.isdir(repo_dir):
    print("cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
else:
    # Existing clone? refresh to the LATEST repo state (fixes land here).
    print("updating existing clone...")
    subprocess.run(["git", "-C", repo_dir, "fetch", "--all"])
    subprocess.run(["git", "-C", repo_dir, "reset", "--hard", "origin/main"])
os.chdir(repo_dir)
print("cwd:", os.getcwd())

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", *args])
    return r.returncode

print("--- 1/3: core dependencies —")
if pip("install", "-r", "requirements.txt") != 0:
    raise AssertionError("مرحلة 1 فشلت — راجع الأخطاء الحمراء أعلاه ثم أعد هذه الخلية")

print("--- 2/3: neucodec بدون تبعياته الثقيلة —")
if pip("install", "--no-deps", "neucodec==0.0.6") != 0:
    raise AssertionError("مرحلة 2 فشلت — راجع الأخطاء أعلاه")

# 3/3: preflight -- verify the EXACT import the server needs, then heal the
# environment if the installed pipecat tree is corrupted (stale/mixed files).
VERIFY_CMD = "from pipecat.pipeline.worker import PipelineParams, PipelineWorker"


def pipecat_ok():
    return subprocess.run(
        [sys.executable, "-c", VERIFY_CMD], capture_output=True
    ).returncode == 0


print("--- 3/3: preflight pipecat import ---")
if pipecat_ok():
    print("pipecat import OK")
else:
    print("WARNING: installed pipecat tree is corrupted (mixed/stale files)")
    print("  -> clean reinstall of pipecat-ai only (deps untouched)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "pipecat-ai"],
        capture_output=True,
    )
    sp = sysconfig.get_paths()["purelib"]
    for p in glob.glob(os.path.join(sp, "pipecat*")):
        print("   purging leftover:", p)
        (shutil.rmtree if os.path.isdir(p) else os.remove)(p)
    if pip("install", "--no-cache-dir", "--no-deps", "pipecat-ai==1.15.0") != 0:
        raise AssertionError(
            "فشل إعادة تثبيت pipecat — راجع الأخطاء الحمراء أعلاه ثم أعد الخلية"
        )
    if not pipecat_ok():
        raise RuntimeError(
            "❌ استيراد pipecat يفشل بعد إعادة التثبيت — أرسل لي المخرجات أعلاه"
        )
    print("pipecat reinstalled and verified OK")

print("✔ Install done — (torchtune مسوّل عنه داخل المشروع فلا يحتاج تنزيله)")

## 3) رمز Hugging Face — **إجباري** 🔑
[`neuphonic/neucodec`](https://huggingface.co/neuphonic/neucodec) مستودع مُقيّد (gated):

1. افتح <https://huggingface.co/neuphonic/neucodec> وسجّل دخول ← زر **Agree and access repository** (مرة واحدة، خلتها فعّلت واضح "You have been granted access").
2. توكن **Read** (classic) من <https://huggingface.co/settings/tokens>.
3. الصقه هنا وشغّل — تفحص الحساب وحق تع عب المستودع فعليًّا.

In [ ]:
#@title 3) Hugging Face token (required for gated codec) {display-mode:"form"}
import os
TOKEN = "" #@param {type:"string"}
TOKEN = TOKEN.strip()
assert TOKEN, "الصق توكن HF من https://huggingface.co/settings/tokens أولًا"
os.environ["HF_TOKEN"] = TOKEN
from huggingface_hub import login, HfApi
login(TOKEN, add_to_git_credential=False)

# Real probes: (1) token validity/binding, (2) actual gated access.
try:
    me = HfApi(token=TOKEN).whoami()
    print("Token OK ✔ — الحساب:", me.get("name"))
except Exception as e:
    print("❌ التوكن غير صالح:")
    print("   أنشئ توكن Read (classic) من https://huggingface.co/settings/tokens وأعد الخلية")
    print("التفاصيل:", str(e)[:250])
    raise

try:
    info = HfApi(token=TOKEN).model_info("neuphonic/neucodec")
    print("Gate OK ✔ — neucodec مفتوح:", info.id)
except Exception as e:
    print("❌ التوكن ما يفتح المستودع المقيّد — الخطوات:")
    print("   1) افتح https://huggingface.co/neuphonic/neucodec")
    print("   2) اضغط 'Agree and access repository' بحساب نفس توكن")
    print("   3) أعد تشغيل هذه الخلية")
    print("التفاصيل:", str(e)[:250])
    raise

In [ ]:
#@title 4) تحميل النماذج مسبقًا (~7 GB) — لازم كلها ✔ بلا ❌ {display-mode:"form"}
from huggingface_hub import snapshot_download

REPOS = ["IFM/Jais-ASR-W4A16-AWQ",        # STT quantized (T4 default)
         "inception42/Jais-3-0.9B",        # chat LLM
         "audarai/Audar-TTS-V1-Flash",     # TTS backbone + voices
         "neuphonic/neucodec"]             # TTS codec (gated -> auth)

for repo in REPOS:
    print("downloading", repo, "...")
    try:
        snapshot_download(repo_id=repo)
        print("  ✔", repo)
    except Exception as e:
        print("  ❌ فشل", repo, "—", str(e)[:300])

# neucodec is REQUIRED: without it TTS cannot decode audio. Second chance,
# then a hard stop with the exact fix (the server would crash without it).
try:
    snapshot_download(repo_id="neuphonic/neucodec")
except Exception as e:
    raise AssertionError(
        "❌ neucodec ما انحمل — الخادم بينهار بدونه. الحل:\n"
        "1) افتح https://huggingface.co/neuphonic/neucodec واضغط 'Agree and access repository'\n"
        "2) توكن Read من https://huggingface.co/settings/tokens وأعد تشغيل خلية 3\n"
        "3) أعد تشغيل هذه الخلية\n"
        "الخطأ: " + str(e)[:400]
    )
print("✔ neucodec cached — تقدر تكمّل بأمان")
print("Model cache step done.")

In [ ]:
#@title 5) تشغيل خادم الصوت على VM كولاب (WebRTC عبر TURN relay) {display-mode:"form"}
APP_PORT = 8080  # internal VM port only; the public URL comes from the tunnel cell (nothing is browsed locally)

import os, subprocess, sys, time, requests
assert os.environ.get("HF_TOKEN"), "شغّل خلية 3 (توكن HF) قبل هذه الخلية"

os.environ["JAIS_ASR_MODEL"] = "auto"          # AWQ checkpoint on T4 (~3.4 GB)
os.environ["JAIS_LLM_DTYPE"] = "fp16"
os.environ["JAIS_TTS_VOICE"] = "demo_female_1"
os.environ["PIPECAT_ICE_SERVERS"] = '[{"urls":["stun:stun.l.google.com:19302","turn:openrelay.metered.ca:80","turn:openrelay.metered.ca:443","turns:openrelay.metered.ca:443?transport=tcp"],"username":"openrelayproject","credential":"openrelayproject"}]'

# Pre-start preflight: the exact import app.py needs. If the VM's installed
# pipecat tree is corrupted (stale/mixed files), rebuild it cleanly.
_preflight = subprocess.run(
    [sys.executable, "-c",
     "from pipecat.pipeline.worker import PipelineParams, PipelineWorker"],
    capture_output=True,
)
if _preflight.returncode != 0:
    print("WARNING: pipecat install on this VM is corrupted — healing...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                    "pipecat-ai"], capture_output=True)
    import glob as _glob, shutil as _shutil, sysconfig as _sysconfig
    sp = _sysconfig.get_paths()["purelib"]
    for p in _glob.glob(os.path.join(sp, "pipecat*")):
        print("   purging leftover:", p)
        (_shutil.rmtree if os.path.isdir(p) else os.remove)(p)
    if subprocess.run([sys.executable, "-m", "pip", "install",
                       "--no-cache-dir", "--no-deps", "pipecat-ai==1.15.0"]
                      ).returncode != 0:
        raise AssertionError("فشل إعادة تثبيت pipecat — راجع الأخطاء أعلاه")
    if subprocess.run([sys.executable, "-c",
                       "from pipecat.pipeline.worker import "
                       "PipelineParams, PipelineWorker"]
                      ).returncode != 0:
        raise RuntimeError("❌ pipecat يفشل بعد إعادة التثبيت — أرسل لي المخرجات")
    print("pipecat healed and verified OK\n")

def log_tail(n=30):
    try:
        return "\n".join(open("/content/jais-voice.log").read().splitlines()[-n:])
    except Exception:
        return "(لا يوجد سجل)"

# Fresh log per start: the tail you see belongs to THIS run only.
logf = open("/content/jais-voice.log", "w")
proc = subprocess.Popen(
    ["python", "app.py", "-t", "webrtc", "--host", "0.0.0.0", "--port", str(APP_PORT)],
    stdout=logf, stderr=logf, cwd="/content/jais-voice-chatbot",
)
print("server started (PID %d) — ننتظر uvicorn..." % proc.pid)

time.sleep(8)
if proc.poll() is not None:
    print("❌ الخادم انهار فور الإقلاع — آخر 30 سطر من السجل:")
    print(log_tail())
    raise RuntimeError("أصلح الخطأ أعلاه من سجل الخادم ثم أعد تشغيل هذه الخلية")

up = False
for _ in range(90):
    if proc.poll() is not None:
        break
    try:
        if requests.get("http://127.0.0.1:%s/docs" % APP_PORT, timeout=2).status_code == 200:
            up = True
            break
    except Exception:
        time.sleep(2)

if not up:
    print("❌ الخادم ما جهز خلال 3 دقائق — آخر 40 سطر من السجل:")
    print(log_tail(40))
    raise RuntimeError("شوف السجل أعلاه — عالج الخطأ وأعد تشغيل الخلية")
print("✔ الخادم راكب — جاهز. شغّل الخلية التالية: رابط الاستضافة العمومية")

In [ ]:
#@title 6) 🌍 استضافة عمومية — رابط تشاركه/تفتحه من أي جهاز {display-mode:"form"}
# Cloudflare quick tunnel: no signup/token; the URL is random on every run.
# Starts ONLY when the local server is healthy, and prints the URL after
# verifying it round-trips through the Cloudflare edge. Uses http2 (TCP)
# because QUIC/UDP is often blocked in sandboxes.
import subprocess, time, re, os, requests

assert "APP_PORT" in globals(), "شغّل خلية 5 قبل هذه الخلية (APP_PORT غير معرّف)"
assert os.environ.get("HF_TOKEN"), "شغّل الخلايا 3 و5 أولًا"

# 0) The local server must answer before we open any tunnel.
server_ok = False
for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:%s/docs" % APP_PORT, timeout=2).status_code == 200:
            server_ok = True
            break
    except Exception:
        time.sleep(2)
if not server_ok:
    print("❌ الخادم ما يستجيب محليًا — آخر 30 سطر من سجل الخادم:")
    try:
        print("\n".join(open("/content/jais-voice.log").read().splitlines()[-30:]))
    except Exception:
        pass
    raise AssertionError("شغّل خلية 5 وانتظر '(✔ الخادم راكب)' قبل الاستضافة")

# 1) Install cloudflared once (single static binary).
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 2) Drop any stale tunnel so the NEW URL is the only live one.
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# 3) Start the tunnel (http2 keeps it alive on UDP-restricted networks).
tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel",
     "--url", "http://localhost:%d" % APP_PORT,
     "--protocol", "http2",
     "--no-autoupdate"],
    stdout=tunnel_log, stderr=tunnel_log,
)

# 4) Parse the URL from the log, then verify it through the Cloudflare edge.
public_url = None
verified = False
for _ in range(45):
    time.sleep(2)
    try:
        text = open("/content/tunnel.log").read()
    except Exception:
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
    if not m:
        continue
    cand = m.group(0)
    public_url = public_url or cand
    try:
        r = requests.get(cand + "/docs", timeout=8)
        if r.status_code < 500:   # edge reachable AND server answering
            public_url, verified = cand, True
            break
    except Exception:
        pass

if public_url:
    print("=" * 62)
    print("🌍 رابط الاستضافة — استخدم هذا فقط (الروابط القديمة تموت):")
    print()
    print("   ", public_url + "/client")
    print()
    if verified:
        print("✔ تم التحقق من الرابط عبر الإنترنت (يرد عليك الآن)")
    else:
        print("⚠ ما قدرنا نتحقق آليًا في الوقت المحدد — جرّبه بالمتصفح؛")
        print("  غالبًا يشتغل بعد لحظات. أو أعِد تشغيل هذه الخلية.")
    print("=" * 62)
    print("ملاحظات: يعمل ما دام كولاب شغّالًا • يتغيّر كل تشغيلة للخلية •")
    print("لو تعطّل لاحقًا شغّل خلية 7 للفحص السريع.")
else:
    print("❌ النفق ما طلّع رابط نهائيًا. آخر سجلاته:")
    try:
        for line in open("/content/tunnel.log").read().splitlines()[-12:]:
            print("   ", line)
    except Exception:
        pass
    print("أعِد تشغيل هذه الخلية (أحيانًا trycloudflare يرفض مؤقتًا)، أو")
    print("أعِد تشغيل هذه الخلية (أحيانًا trycloudflare يرفض مؤقتًا).")

In [ ]:
#@title 7) فحص تشخيصي سريع: ليش الرابط العام ما اشتغل؟ {display-mode:"form"}
import subprocess, re, requests

try:
    r = requests.get("http://127.0.0.1:%s/docs" % APP_PORT, timeout=3)
    print("1) الخادم محليًا:", "✔ راكب (HTTP 200)" if r.status_code == 200
          else "⚠ استجاب برمز %s" % r.status_code)
except Exception as e:
    print("1) الخادم محليًا: ❌ ما يستجيب → شغّل خلية 5 (", str(e)[:100], ")")

alive = subprocess.run(["pgrep", "-f", "cloudflared"], capture_output=True).returncode == 0
print("2) عملية النفق:", "✔ حيّة" if alive else "❌ مقطوعة → أعِد تشغيل خلية 6")

try:
    text = open("/content/tunnel.log").read()
    urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
    cur = urls[-1] if urls else None
    print("3) آخر رابط سجّله النفق:", cur or "— لا يوجد → أعِد تشغيل خلية 6")
    if cur:
        try:
            rr = requests.get(cur + "/docs", timeout=10)
            if rr.status_code == 200:
                print("   ↳ عبر الإنترنت: ✔ يشتغل الآن (200) — الرابط سليم؛")
                print("     جرّب متصفحًا/شبكة أخرى لو ما فتح عندك")
            elif rr.status_code >= 500:
                print("   ↳ عبر الإنترنت: النفق موصول لكن الخادم يرجع %s →" % rr.status_code)
                print("     شغّل خلية 5 ثم خلية 6 من جديد")
            else:
                print("   ↳ عبر الإنترنت: HTTP", rr.status_code, "— جرب الرابط مرة ثانية")
        except Exception as e:
            print("   ↳ عبر الإنترنت: ❌ ما وصّل:", str(e)[:100], "→ أعِد خلية 6")
except FileNotFoundError:
    print("3) لا يوجد سجل نفق → شغّل خلية 6 أولًا")

print()
print("--- آخر 15 سطر من سجل النفق ---")
try:
    print("\n".join(open("/content/tunnel.log").read().splitlines()[-15:]))
except Exception:
    pass

print("--- آخر 20 سطر من سجل الخادم ---")
!tail -20 /content/jais-voice.log

## Troubleshooting
| العلامة | السبب | الحل |
|---|---|---|
| مرحلة pip فشلت في خلية 2 | إشاعرة الاعتمادات | الأحمر أعلاه يشرح؛ أعد الخلية (تعيد نفس المرحلة) |
| الخادم انهار وتطبع السجل داخل الخلية | خطأ إقلاع (تبعية/كود) | انسخ **أول Traceback** وأرسله لنا |
| `This site can't be reached` / 1033/530 | النفق انطفأ/الكولاب انقطع | خلية 6 من جديد (كل مرّة رابط جديد) |
| **502 Bad Gateway** | النفق حي والخادم وقف | خلية 5 ثم خلية 6 |
| الصفحة تفتح و **Connect يلف** | أول اتصال يسخّن النماذج | انتظر ~دقيقة، ثم خلية 7 |
| `❌` في خلية 4 على neucodec | الترخيص/التوكن | صفحة المستودع ← Agree ← توكن Read ← خلية 3 |
| الصوت ما يوصل بعد الاتصال | NAT/الوسائط | TURN عبر TCP — أعد الاتصال وتشيك خلية 7 |
| OOM | الذاكرة | `JAIS_ASR_MODEL=auto` + قلل `JAIS_LLM_MAX_NEW_TOKENS` |
| الردود بطيئة على T4 | متوقع | ASR 6B مضغوط → 3-8 ثوان/جملة |

سجلات: `!tail -50 /content/jais-voice.log` — النفق: `!tail -30 /content/tunnel.log`